# Silver → Gold | CineData Analytics

**1. Modelagem **

- **Fato** (`fact_movies_performance`): os **números** (dinheiro, notas, votos). 1 linha por filme.
- **Dimensões** (`dim_*`): as **descrições** (título, gênero, pessoa, produtora...). Respondem "quem/o quê/quando".
- **Pontes** (`bridge_*`): resolvem relação **muitos-para-muitos**. Um filme tem vários gêneros e um gênero tem vários filmes. Sem a ponte, ligar as tabelas duplicaria as linhas da fato e somaria a receita várias vezes.
- **Surrogate Key (`sk_*`)**: um número simples criado por nós para identificar cada linha. Ligar tabelas por número é mais rápido e estável do que por texto.

**2. Tabela de contexto para IA** (`gold_genai_movies_context`): um texto corrido por filme para o Vector Search do time de IA.

**3. Desafio de Analytics**: as 6 perguntas de negócio.

## 0. Configurações

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"
BANCO_SILVER = "silver"
BANCO_GOLD = "gold"

#   False (padrão) -> a fato tem TODOS os filmes de dim_movies (a pergunta 2, popularidade, considera todos)
#   True           -> a fato só tem filmes com status "Lançado"

SOMENTE_LANCADOS = False

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BANCO_GOLD}")

DataFrame[]

In [0]:
def silver(tabela: str):
    return spark.table(f"{BANCO_SILVER}.{tabela}")


def gerar_sk(df, nome_sk: str, ordenar_por: list):
    """
    Cria a Surrogate Key: row_number() numera as linhas 1, 2, 3... seguindo a ordenação pedida.
    Ordenar pela chave natural, pois deixa a numeração previsível.
    O resultado é convertido para BIGINT, como pede o enunciado.
    As chaves são regeneradas a cada execução, mas TODAS as tabelas Gold são refeitas juntas
    no mesmo notebook, então as ligações (fato, pontes) sempre ficam consistentes.
    """
    return df.withColumn(nome_sk, F.row_number().over(Window.orderBy(*ordenar_por)).cast("bigint"))


def salvar_gold(df, tabela: str) -> None:
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BANCO_GOLD}.{tabela}")
    )
    print(f"OK  {BANCO_GOLD}.{tabela}  |  {spark.table(f'{BANCO_GOLD}.{tabela}').count()} linhas")

## 1. Dimensões

### `gold.dim_movies`
Uma linha por filme, com os dados descritivos. É o centro do esquema: fato e pontes se ligam a ela pela `sk_movie_id`.

In [0]:
dim_movies = gerar_sk(silver("tb_info_filmes"), "sk_movie_id", ["id_filme"]).select(
    F.col("sk_movie_id").cast("bigint"),
    F.col("id_filme").cast("string"),              # chave natural (a que vinha da origem)
    F.col("titulo").cast("string"),
    F.col("data_lancamento").cast("date"),
    F.col("ano_lancamento").cast("int"),
    F.col("duracao_minutos").cast("int"),
    F.col("idioma_original").cast("string"),
    F.col("status_filme").cast("string"),
    F.col("sinopse").cast("string"),
)
salvar_gold(dim_movies, "dim_movies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.dim_movies  |  97611 linhas


### `gold.dim_genres`, `gold.dim_people`, `gold.dim_companies`
Catálogos únicos (sem repetição). `dim_people` guarda só pessoas físicas (Ator, Diretor, Roteirista); as produtoras vão para `dim_companies`.

Uma mesma pessoa pode ser Ator e Diretor. Como `dim_people` tem uma coluna `tipo_pessoa`, ela aparece uma vez para cada papel (a chave é o par nome + tipo).

In [0]:
# --- dim_genres ---
dim_genres = gerar_sk(
    silver("tb_generos").select(F.col("genero").alias("nome_genero")).distinct(),
    "sk_genre_id", ["nome_genero"],
).select(F.col("sk_genre_id").cast("bigint"), F.col("nome_genero").cast("string"))
salvar_gold(dim_genres, "dim_genres")

# --- dim_people --- (Ator, Diretor, Roteirista)
dim_people = gerar_sk(
    silver("tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct(),
    "sk_person_id", ["tipo_pessoa", "nome_pessoa"],
).select(
    F.col("sk_person_id").cast("bigint"),
    F.col("nome_pessoa").cast("string"),
    F.col("tipo_pessoa").cast("string"),
)
salvar_gold(dim_people, "dim_people")

# --- dim_companies --- (Produtora)
dim_companies = gerar_sk(
    silver("tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct(),
    "sk_company_id", ["nome_produtora"],
).select(F.col("sk_company_id").cast("bigint"), F.col("nome_produtora").cast("string"))
salvar_gold(dim_companies, "dim_companies")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.dim_genres  |  610 linhas
OK  gold.dim_people  |  418674 linhas
OK  gold.dim_companies  |  46035 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### `gold.dim_reviews`
As avaliações individuais viram um resumo por filme: quantas avaliações recebeu e a nota média (arredondada em 2 casas).
`avg()` ignora `NULL`, então avaliações com nota inválida contam na quantidade mas não entram na média.

In [0]:
resumo_reviews = (
    silver("tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
)

# inner join: só ficam avaliações de filmes que existem em dim_movies (garante que a FK sempre aponta para um filme válido)
dim_reviews = gerar_sk(
    resumo_reviews.join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner"),
    "sk_review_id", ["sk_movie_id"],
).select(
    F.col("sk_review_id").cast("bigint"),
    F.col("sk_movie_id").cast("bigint"),
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios",
)
salvar_gold(dim_reviews, "dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.dim_reviews  |  27226 linhas


## 2. Tabela fato: `gold.fact_movies_performance`

**Grão:** 1 linha por filme. Partimos de `dim_movies` e trazemos financeiro e engajamento com LEFT JOIN:
- `LEFT` garante que nenhum filme some só porque não tem dado financeiro ou de métricas;
- como `tb_financeiro_filmes` e `tb_metricas_engajamento` já são únicas por filme (deduplicadas na Silver), o join não duplica linhas.

Na última linha da célula há uma checagem automática do grão.

In [0]:
base_fato = dim_movies.select("sk_movie_id", "id_filme", "status_filme")
if SOMENTE_LANCADOS:
    base_fato = base_fato.filter(F.col("status_filme") == "Lançado")

fact_movies_performance = (
    base_fato.select("sk_movie_id", "id_filme")
    .join(silver("tb_financeiro_filmes"), "id_filme", "left")
    .join(silver("tb_metricas_engajamento"), "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),                        # FK -> dim_movies
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)
salvar_gold(fact_movies_performance, "fact_movies_performance")

# Checagem do grão: linhas == filmes distintos (senão algum join duplicou)
n_linhas = fact_movies_performance.count()
n_filmes = fact_movies_performance.select("sk_movie_id").distinct().count()
assert n_linhas == n_filmes, f"Grão quebrado na fato: {n_linhas} linhas para {n_filmes} filmes"
print("Grão OK: 1 linha por filme")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.fact_movies_performance  |  97611 linhas
Grão OK: 1 linha por filme


## 3. Tabelas-ponte (bridge)

Cada ponte é só uma lista de **pares** (filme, gênero/pessoa/produtora). Para montar, trocamos os nomes da Silver
pelas chaves substitutas: `id_filme → sk_movie_id`, `genero → sk_genre_id`, e assim por diante. O `distinct()` garante que o mesmo par não se repita.

In [0]:
filmes_sk = dim_movies.select("sk_movie_id", "id_filme")

# --- bridge_movie_genre ---
bridge_movie_genre = (
    silver("tb_generos")
    .join(filmes_sk, "id_filme", "inner")
    .join(dim_genres, F.col("genero") == F.col("nome_genero"), "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_genre_id").cast("bigint"))
    .distinct()
)
salvar_gold(bridge_movie_genre, "bridge_movie_genre")

# --- bridge_movie_person --- (o par nome + tipo identifica a pessoa em dim_people)
bridge_movie_person = (
    silver("tb_pessoas_empresas")
    .join(filmes_sk, "id_filme", "inner")
    .join(
        dim_people,
        (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        "inner",
    )
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_person_id").cast("bigint"))
    .distinct()
)
salvar_gold(bridge_movie_person, "bridge_movie_person")

# --- bridge_movie_company ---
bridge_movie_company = (
    silver("tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(filmes_sk, "id_filme", "inner")
    .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"), "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_company_id").cast("bigint"))
    .distinct()
)
salvar_gold(bridge_movie_company, "bridge_movie_company")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.bridge_movie_genre  |  140533 linhas
OK  gold.bridge_movie_person  |  758317 linhas
OK  gold.bridge_movie_company  |  118865 linhas


## 4. Entrega 2: `gold.gold_genai_movies_context` (base para o assistente de IA / RAG)

**Objetivo:** um texto corrido por filme, para o time de IA vetorizar (Vector Search).

O `concat()` devolve `NULL` se qualquer pedaço for `NULL`. Um filme sem diretor ou sem sinopse ficaria com o documento inteiro `NULL` e sumiria do assistente em silêncio.

**A solução:** `coalesce(campo, 'texto de fallback')` em cada campo que pode vir nulo, antes de concatenar.
`coalesce` devolve o primeiro valor não nulo da lista.

| Campo | Pode vir nulo? | Fallback usado |
|---|---|---|
| título | Sim (pouco provável) | "título não informado" |
| ano | Sim (data de lançamento inválida) | "ano não informado" |
| receita / orçamento | Sim, muito (zero e "Unknown" viraram NULL na Silver) | "um valor não informado" |
| atores | Sim | "elenco não informado" |
| diretor | Sim | "diretor não informado" |
| sinopse | Sim | "sinopse não disponível" |

Outros cuidados:
- "Atores principais" = os 5 primeiros do elenco (a ordem original é preservada pela coluna `posicao` da Silver).
- Vários atores/diretores viram uma string com `collect_list` + `array_join`.
- Valores em dólar formatados com `format_number` ("US$ 1,234,567.00").
- Quebras de linha e espaços repetidos da sinopse são normalizados (melhor para vetorização).

In [0]:
MAX_ATORES_PRINCIPAIS = 5

# Nomes de cada filme via PONTE + DIMENSÃO (como pede o enunciado), com a posição vinda da Silver para ordenar o elenco.
pessoas_por_filme = (
    bridge_movie_person
    .join(dim_people, "sk_person_id", "inner")
    .join(filmes_sk, "sk_movie_id", "inner")
    .join(
        silver("tb_pessoas_empresas").select(
            "id_filme",
            F.col("nome_entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa"),
            "posicao",
        ),
        ["id_filme", "nome_pessoa", "tipo_pessoa"],
        "left",
    )
)

def agregar_nomes(df):
    """
    Junta os nomes de cada filme em UMA string, respeitando a ordem de `posicao`:
      collect_list  -> junta os (posicao, nome) em uma lista
      sort_array    -> ordena pela posicao (primeiro campo do struct)
      transform     -> fica só com o nome
      array_join    -> junta tudo com ", "
    """
    return df.groupBy("sk_movie_id").agg(
        F.array_join(
            F.transform(F.sort_array(F.collect_list(F.struct("ordem", "nome_pessoa"))), lambda x: x["nome_pessoa"]),
            ", ",
        ).alias("texto")
    )

ordem_no_filme = Window.partitionBy("sk_movie_id", "tipo_pessoa").orderBy(F.col("posicao").asc_nulls_last(), "nome_pessoa")
pessoas_por_filme = pessoas_por_filme.withColumn("ordem", F.row_number().over(ordem_no_filme))

atores = agregar_nomes(
    pessoas_por_filme.filter((F.col("tipo_pessoa") == "Ator") & (F.col("ordem") <= MAX_ATORES_PRINCIPAIS))
).withColumnRenamed("texto", "atores_principais")

diretores = agregar_nomes(
    pessoas_por_filme.filter(F.col("tipo_pessoa") == "Diretor")
).withColumnRenamed("texto", "diretores")

# Montagem do documento 
# LEFT JOIN a partir de dim_movies: todo filme entra na tabela de contexto, mesmo sem ator, diretor ou financeiro.
contexto_base = (
    dim_movies
    .join(fact_movies_performance.select("sk_movie_id", "receita_usd", "orcamento_usd"), "sk_movie_id", "left")
    .join(atores, "sk_movie_id", "left")
    .join(diretores, "sk_movie_id", "left")
)

titulo_txt = F.coalesce(F.col("titulo"), F.lit("título não informado"))
ano_txt = F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado"))

# concat devolve NULL se o format_number for NULL, e aí o coalesce entra com o texto de fallback
receita_txt = F.coalesce(F.concat(F.lit("US$ "), F.format_number("receita_usd", 2)), F.lit("um valor não informado"))
orcamento_txt = F.coalesce(F.concat(F.lit("US$ "), F.format_number("orcamento_usd", 2)), F.lit("um valor não informado"))

atores_txt = F.coalesce(F.col("atores_principais"), F.lit("elenco não informado"))
diretor_txt = F.coalesce(F.col("diretores"), F.lit("diretor não informado"))

# Sinopse: espaços/quebras de linha repetidos viram 1 espaço; ponto final removido (o template já coloca um ponto no fim)
sinopse_limpa = F.regexp_replace(F.regexp_replace(F.trim(F.col("sinopse")), r"\s+", " "), r"[.\s]+$", "")
sinopse_txt = F.coalesce(F.when(sinopse_limpa != "", sinopse_limpa), F.lit("sinopse não disponível"))

documento = F.concat(
    F.lit("O filme "), titulo_txt,
    F.lit(", lançado no ano de "), ano_txt,
    F.lit(", faturou "), receita_txt,
    F.lit(" e teve um custo de "), orcamento_txt,
    F.lit(". Estrelado por "), atores_txt,
    F.lit(" e dirigido por "), diretor_txt,
    F.lit(", o filme possui a seguinte sinopse: "), sinopse_txt,
    F.lit("."),
)

genai_movies_context = contexto_base.select(
    F.col("id_filme").alias("movie_id"),
    titulo_txt.alias("title"),
    documento.alias("llm_context_document"),
)
salvar_gold(genai_movies_context, "gold_genai_movies_context")

# Validações: nenhum filme pode ter sumido nem ficar com documento NULL
total_dim = dim_movies.count()
total_ctx = spark.table(f"{BANCO_GOLD}.gold_genai_movies_context").count()
docs_nulos = spark.table(f"{BANCO_GOLD}.gold_genai_movies_context").filter(F.col("llm_context_document").isNull()).count()
print(f"Filmes em dim_movies: {total_dim} | Documentos de contexto: {total_ctx} | Documentos NULL: {docs_nulos}")
assert total_dim == total_ctx and docs_nulos == 0, "Algum filme sumiu ou ficou com documento NULL"

display(spark.table(f"{BANCO_GOLD}.gold_genai_movies_context").limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK  gold.gold_genai_movies_context  |  97611 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Filmes em dim_movies: 97611 | Documentos de contexto: 97611 | Documentos NULL: 0


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou um valor não informado e teve um custo de um valor não informado. Estrelado por Izzy Jones, Erika Alexander, Aron Von Andrian, Steven Michael-o’hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou um valor não informado e teve um custo de um valor não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: sinopse não disponível."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou um valor não informado e teve um custo de um valor não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou um valor não informado e teve um custo de um valor não informado. Estrelado por Marcello Urgeghe, João Pedro Bénard, Isabel Abreu, Inês Pronto e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou um valor não informado e teve um custo de um valor não informado. Estrelado por Soulayman Rkiba, Gabrielle Cohen, Claire Chust, Maxime Pambet, Biyouna e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: sinopse não disponível."


## 5. Checagem de integridade do modelo

Chaves primárias únicas e nenhuma chave estrangeira "órfã" (apontando para algo que não existe).

In [0]:
def tabela(nome):
    return spark.table(f"{BANCO_GOLD}.{nome}")

# 1) PK única
for nome_tab, pk in [("dim_movies", "sk_movie_id"), ("dim_genres", "sk_genre_id"), ("dim_people", "sk_person_id"),
                     ("dim_companies", "sk_company_id"), ("dim_reviews", "sk_review_id"),
                     ("fact_movies_performance", "sk_movie_id")]:
    t = tabela(nome_tab)
    assert t.count() == t.select(pk).distinct().count(), f"PK duplicada em {nome_tab}"
print("PKs únicas: OK")

# 2) FK órfã (left_anti = linhas da esquerda que NÃO têm par na direita; esperado = 0)
checagens = [
    ("fact_movies_performance", "dim_movies", "sk_movie_id"),
    ("dim_reviews", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_genres", "sk_genre_id"),
    ("bridge_movie_person", "dim_movies", "sk_movie_id"),
    ("bridge_movie_person", "dim_people", "sk_person_id"),
    ("bridge_movie_company", "dim_movies", "sk_movie_id"),
    ("bridge_movie_company", "dim_companies", "sk_company_id"),
]
for origem, destino, chave in checagens:
    orfas = tabela(origem).join(tabela(destino), chave, "left_anti").count()
    assert orfas == 0, f"{origem}.{chave} tem {orfas} chaves órfãs"
print("FKs sem órfãs: OK")

PKs únicas: OK
FKs sem órfãs: OK


## 6. Desafio de Analytics (perguntas de negócio)

Todas as respostas usam **apenas** tabelas da camada Gold.

### Pergunta 1: Qual é a receita total (em R$) de todos os filmes da base?

`sum()` ignora `NULL`, então filmes sem receita informada simplesmente não entram na soma.

In [0]:
display(
    spark.table(f"{BANCO_GOLD}.fact_movies_performance")
    .agg(F.sum("receita_brl").alias("receita_total_brl"))
)

receita_total_brl
822039420717.37


### Pergunta 2: Quais são os 5 filmes com maior popularidade?

In [0]:
fato = spark.table(f"{BANCO_GOLD}.fact_movies_performance")
dim_m = spark.table(f"{BANCO_GOLD}.dim_movies")

display(
    fato.join(dim_m, "sk_movie_id")
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())     # desc coloca NULL por último
    .limit(5)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0


### Pergunta 3: Quantos filmes cada gênero possui? (do maior para o menor)

`countDistinct` por filme garante que um filme conte só uma vez em cada gênero.

In [0]:
display(
    spark.table(f"{BANCO_GOLD}.bridge_movie_genre")
    .join(spark.table(f"{BANCO_GOLD}.dim_genres"), "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), "nome_genero")
)

nome_genero,qtd_filmes
Drama,32127
Documentary,18927
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
Tv Movie,4066


### Pergunta 4: Os 10 filmes de maior receita, com título, receita (US$ e R$) e ranking

`RANK()` dá a mesma posição a filmes empatados e "pula" a numeração seguinte (1, 2, 2, 4...). Por isso, em caso de empate na 10ª posição, podem aparecer mais de 10 linhas.

In [0]:
janela_ranking = Window.orderBy(F.col("receita_usd").desc())

top_receita = (
    fato.filter(F.col("receita_usd").isNotNull())
    .join(dim_m, "sk_movie_id")
    .withColumn("ranking", F.rank().over(janela_ranking))
    .filter(F.col("ranking") <= 10)
    .select("ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("ranking")
)
display(top_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14311080000.00
2,Avatar: The Way of Water,2320250281.00,11859031211.22
3,AVENGERS: INFINITY WAR,2052415039.00,10490098505.83
4,spider-man: no way home,1921847111.00,9822752769.03
5,The Lion King,1663075401.00,8500144682.05
6,Top Gun: Maverick,1488732821.00,7609062321.41
7,Barbie,1428545028.00,7301436492.61
8,The Super Mario Bros. Movie,1355725263.00,6929247391.72
9,Black Panther,1349926083.00,6899607202.82
10,Star Wars: The Last Jedi,1332698830.00,6811556990.01


### Data limite para as perguntas 5 e 6

O enunciado pede que "últimos 2/5 anos" seja contado a partir da **data de lançamento mais recente da base**,
ignorando datas futuras e filmes não lançados. Então:
- consideramos só `status_filme = 'Lançado'` e `data_lancamento <= hoje`;
- a data limite é o `max(data_lancamento)` desse recorte;
- a janela é `(data limite - N anos, data limite]`.

In [0]:
data_limite = (
    dim_m.filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento"))
    .first()[0]
)
print("Data de lançamento mais recente (data limite):", data_limite)

def filmes_na_janela(anos: int):
    """Filmes lançados nos últimos `anos` anos, contados a partir da data limite."""
    inicio = F.add_months(F.lit(data_limite), -12 * anos)
    return dim_m.filter((F.col("data_lancamento") > inicio) & (F.col("data_lancamento") <= F.lit(data_limite)))

Data de lançamento mais recente (data limite): 2026-02-19


### Pergunta 5: Qual ator teve mais participações em filmes lançados nos últimos 2 anos?

A 1ª linha é a resposta; mostramos o top 10 para enxergar empates.

In [0]:
display(
    filmes_na_janela(2).select("sk_movie_id")
    .join(spark.table(f"{BANCO_GOLD}.bridge_movie_person"), "sk_movie_id")
    .join(spark.table(f"{BANCO_GOLD}.dim_people").filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc(), "nome_pessoa")
    .limit(10)
)

nome_pessoa,qtd_participacoes
Kevin Hart,61
Nathalie Emmanuel,60
Ben Schwartz,26
John Cena,26
Paula Pell,22
Greg Kriek,15
Melissa Ponzio,14
Adlih Torres,12
Cooper Tomlinson,12
Gloria Karel,12


### Pergunta 6: Qual produtora teve o maior lucro nos últimos 5 anos?

O lucro de um filme coproduzido é atribuído por inteiro a cada produtora (não há dado para dividir proporcionalmente).
Mostramos o lucro em US$ e em R$.

In [0]:
display(
    filmes_na_janela(5).select("sk_movie_id")
    .join(fato.select("sk_movie_id", "lucro_usd", "lucro_brl"), "sk_movie_id")
    .join(spark.table(f"{BANCO_GOLD}.bridge_movie_company"), "sk_movie_id")
    .join(spark.table(f"{BANCO_GOLD}.dim_companies"), "sk_company_id")
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"), F.sum("lucro_brl").alias("lucro_total_brl"))
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(10)
)

nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5772329679.00,29502954222.35
Marvel Studios,4953462823.00,25317643834.63
Columbia Pictures,3662050755.00,18717107613.88
Pascal Pictures,2701952454.00,13809949187.64
Illumination,2431353473.00,12426890735.85
20th Century Studios,2212815245.00,11309919998.73
Paramount,2169053883.00,11086251301.41
Kevin Feige Productions,2138205367.00,10928581451.27
Lightstorm Entertainment,1860250281.00,9507925211.22
Heyday Films,1635495727.00,8359182210.27
